<a href="https://colab.research.google.com/github/plrushe/Honours-Code/blob/main/ROBERTA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip uninstall -y numpy pandas
!{sys.executable} -m pip install -q -U "transformers==4.44.2" "datasets==2.21.0" "evaluate==0.4.2" "accelerate==0.33.0"
!{sys.executable} -m pip install -q -U "scikit-learn==1.4.2" "pandas==2.2.2" "numpy==1.26.4"

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print("torch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

set_seed(42)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
dataset = pd.read_csv("patient_diaries.csv")
dataset = dataset[["text", "label"]].copy()

norm = dataset["label"].astype(str).str.strip().str.lower()
mapping = {"no depression": 0, "depression": 1}
dataset["label"] = norm.map(mapping)

dataset = dataset.dropna(subset=["text", "label"])
dataset["label"] = dataset["label"].astype("int64")
dataset["text"] = dataset["text"].astype(str)

print("Dataset shape:", dataset.shape)
print(dataset["label"].value_counts())
dataset.head()


Dataset shape: (1000, 2)
label
0    514
1    486
Name: count, dtype: int64


,text,label
0,I felt happy today. I shared a laugh with a fr...,0
1,I felt fine today. I wrote in my journal and e...,1
2,I felt confused today. I felt overwhelmed with...,1
3,I felt confused today. I felt a glimmer of hop...,0
4,I felt down today. I felt a glimmer of hope to...,0


In [ ]:
def basic_clean(text: str) -> str:
    text = text.strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

dataset["text"] = dataset["text"].apply(basic_clean)

In [ ]:
train_df, test_df = train_test_split(
    dataset,
    test_size=0.2,
    stratify=dataset["label"],
    random_state=42
)
print("Train:", train_df.shape, "Test:", test_df.shape)

Train: (800, 2) Test: (200, 2)


In [ ]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).rename_column("label", "labels")
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True)).rename_column("label", "labels")


In [ ]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

MAX_LEN = 256

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_batch, batched=True)
test_ds  = test_ds.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)

training_args = TrainingArguments(
    output_dir="roberta_out",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,     # good starting point
    weight_decay=0.01,

    logging_steps=50,
    report_to="none",
    seed=42,
)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.434100,0.249578,0.890000,0.886598,0.886598,0.886598
2,0.200700,0.234918,0.895000,0.963415,0.814433,0.882682
3,0.168200,0.221521,0.910000,0.883495,0.938144,0.910000
4,0.166000,0.231283,0.910000,0.883495,0.938144,0.910000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=200, training_loss=0.2422606086730957, metrics={'train_runtime': 1041.4241, 'train_samples_per_second': 3.073, 'train_steps_per_second': 0.192, 'total_flos': 34533326016000.0, 'train_loss': 0.2422606086730957, 'epoch': 4.0})

In [ ]:
pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)

print("Hold-out test results")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1:", f1_score(y_true, y_pred, zero_division=0))

print("\nClassification report:\n", classification_report(y_true, y_pred, digits=4))
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred))


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Hold-out test results
Accuracy: 0.91
Precision: 0.883495145631068
Recall: 0.9381443298969072
F1: 0.91

Classification report:
               precision    recall  f1-score   support

           0     0.9381    0.8835    0.9100       103
           1     0.8835    0.9381    0.9100        97

    accuracy                         0.9100       200
   macro avg     0.9108    0.9108    0.9100       200
weighted avg     0.9116    0.9100    0.9100       200


Confusion matrix:
 [[91 12]
 [ 6 91]]


In [ ]:
SAVE_DIR = "models/roberta_depression_classifier"
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved to:", SAVE_DIR)